# 计算LLaMA模型的计算量

假设输入seq_len=100, 目标的seq_len=128~4096, vocab_size=32000, hidden_size=3072, num_heads=24, num_layers=32

## 1. 计算参数量

In [1]:
import numpy as np
import pandas as pd

seq_len = 100
vocab_size = 32000
hidden_size = 4096
mlp_size = 11008
num_heads = 24
num_layers = 32

token_embedding = vocab_size * hidden_size
query_weight = hidden_size * hidden_size
key_weight = hidden_size * hidden_size
value_weight = hidden_size * hidden_size
output_weight = hidden_size * hidden_size
mlp = hidden_size * mlp_size * 3
ln = hidden_size * 2
lm_head = hidden_size * vocab_size

total = token_embedding + (query_weight + key_weight + value_weight + output_weight + mlp + ln) * num_layers + lm_head

print(f"Total params: {total / 1000000000:,}B")

Total params: 6.73841152B


## 2. 计算量统计

In [2]:
total = 0
d_h = hidden_size // num_heads

# prefill
token_embedding = seq_len * hidden_size * vocab_size
query_project = hidden_size * hidden_size * seq_len
key_project = hidden_size * hidden_size * seq_len
value_project = hidden_size * hidden_size * seq_len
qkt = num_heads * seq_len * seq_len * d_h
pv = num_heads * seq_len * d_h * seq_len
output_project = hidden_size * hidden_size * seq_len
mlp = hidden_size * mlp_size * seq_len * 3

total += token_embedding + (query_project + key_project + value_project + qkt + pv + output_project + mlp) * num_layers
print(f"Prefill FLOPS: {total / 1000000000:,}GFLOPS")

# decode
infer_time = {}
seq_len_list = [128, 256, 512, 1024, 2048, 4096]

for s in seq_len_list:
    time = total
    for i in range(seq_len, s):
        token_embedding = hidden_size * vocab_size
        query_project = hidden_size * hidden_size
        key_project = hidden_size * hidden_size
        value_project = hidden_size * hidden_size
        qkt = num_heads * i * d_h
        pv = num_heads * d_h * i
        output_project = hidden_size * hidden_size
        mlp = hidden_size * mlp_size * 3

        time += token_embedding + (query_project + key_project + value_project + qkt + pv + output_project + mlp) * num_layers
    infer_time[s] = time / 1000000000

print("Inference time:")
pd.DataFrame(infer_time.items(), columns=["seq_len", "GFLOPS"])

Prefill FLOPS: 663.3189376GFLOPS
Inference time:


,seq_len,TFLOPS
0,128,849.146943
1,256,1701.253421
2,512,3418.300946
3,1024,6903.734278
4,2048,14079.954065
5,4096,29253.806135
